In [1]:
import numpy as np
import h5py
from tenpy.tools.hdf5_io import Hdf5Loader
import pandas as pd
import glob
import signac
from plotnine import *
from datetime import datetime
project = signac.get_project("../../../",)
project.detect_schema()

import matplotlib.pyplot as plt
# Enable external LaTeX rendering via Matplotlib rcParams'
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}" # Optional: load extra packages
})
%config InlineBackend.figure_format = 'svg'


prl_theme = theme(
    # A. Canvas Size: Set it here so you see the 'real' result in your IDE
    figure_size=(3.375, 1.8), # 3.375" width, ~1.8" height
    plot_background=element_blank(),
    plot_margin=0,
    #legend_position='bottom',  # X and Y inside the plot area
    legend_background=element_rect(fill='none'),
    #legend_background=element_rect(fill='white', color='black'),
    legend_text=element_text(size=7, margin={'t':2}),       # size in points
    legend_title=element_text(size=8),
    legend_spacing=0.05,
    #legend_position='top',        # CRITICAL: Move legend to top
    legend_direction='horizontal',# Horizontal bar takes less width
    legend_box_margin=0,          # Remove padding
    legend_key_height=4,          # Thin colorbar (units are pixels/points)
    legend_key_width=55,          # Short colorbar

    axis_title=element_text(size=8),
    axis_text=element_text(size=7, angle=45),
    panel_background=element_blank(),
    axis_line=element_line(color='black'),
    legend_key=element_blank(),
    text=element_text(size=8),

    #strip_text=element_text(size=6,  margin={'t': 1, 'b': 1, 'l':1, 'r':1})

)


In [2]:
jobs = ['ed312054ffb1df312e55895d4250c12c', '1338435365a19c04a64539d7e75e995e', 'c1ee62112d7ed57d2e76d6fb2e116255', 'c0d1499fa767a08d52b42412152bce2e','fb13575a2ea82785862a1dae1795f2c9']

In [3]:
df_params = pd.DataFrame()
for current_job in jobs:
    current_job = project.open_job(id=current_job)
    df_current = project.find_jobs(current_job.statepoint).to_dataframe(flatten=True)
    df_params = pd.concat([df_current, df_params])
df_params['a'] = df_params['sp.model_params.a']
df_params['g'] = df_params['sp.model_params.g']
df_params['Nf_num'] = df_params['sp.model_params.Nf']
df_params['Nf'] = pd.Categorical(df_params['sp.model_params.Nf'], categories=[3,2,1], ordered=True)
df_params['Nr_num'] = df_params['sp.model_params.Nr']
df_params['$\\theta$'] = df_params['sp.model_params.theta'].apply(lambda x: "$\pi$" if np.abs(x-np.pi)<1e-10 else "0")
df_params['\\theta'] = df_params['sp.model_params.theta'].apply(lambda x: "\pi" if np.abs(x-np.pi)<1e-10 else "0")
df_params['m'] = df_params['sp.model_params.m'].apply(lambda x: '(' + ','.join([str(i) for i in x]) + ')')
df_params['diag_method'] = df_params['sp.algorithm_params.diag_method'].apply(lambda x: x[0])
#df_params['charge_mod'] = df_params['sp.model_params.charge_mod'].apply(lambda x: 0 if x is None else x[0])
df_params['L'] =df_params['sp.model_params.L']

<>:11: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:12: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:11: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:12: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
/tmp/ipykernel_3647722/321494200.py:11: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
/tmp/ipykernel_3647722/321494200.py:12: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.


In [4]:
df = {'entropy': [], 'energy':[], 'r':[]}
df_fitted = {'entropy':[], 'r':[]}
df_predicted = {'entropy':[], 'r':[]}
df_labels = {'label':[], 'entropy':[], 'r':[]}
indices = []
crit_point = "Ising"
for current_job in jobs:
    job = project.open_job(id=current_job)
    for res_file in glob.glob(job.fn('results_simulation.h5')):
        with h5py.File(job.fn(res_file), 'r') as f:
            EE = Hdf5Loader(f['measurements']['entropy']).load()
            E = Hdf5Loader(f['energy']).load()
            GS = Hdf5Loader(f['measurements']['Gausses_law']).load()
            df['entropy'].append(EE[-1, :])
            df['energy'].append(E)
            print(job.id)
            indices.append(job.id)
    for res_file in glob.glob(job.fn('results_central_charge_per_unit.h5')):
        with h5py.File(job.fn(res_file), 'r') as f:
            EE_fit = Hdf5Loader(f['fitted_entropies']).load()
            r_fit = Hdf5Loader(f['fitted_cuts']).load()
            EE_pred = Hdf5Loader(f['model_preditions']).load()
            r_all  = Hdf5Loader(f['all_cuts']).load()
            params = Hdf5Loader(f['params']).load()
            cov = Hdf5Loader(f['cov']).load()
            df['r'].append(r_all)
            df_fitted['entropy'].append(EE_fit)
            df_fitted['r'].append(r_fit)
            df_predicted['entropy'].append(EE_pred)
            df_predicted['r'].append(r_all)
            df_labels['entropy'].append(0.5*(np.max(EE)-np.min(EE))+ np.min(EE))
            df_labels['r'].append(0.5*(np.max(r_all)-np.min(r_all)) + np.min(r_all))
            df_labels['label'].append(f"$c_{{ \\rm {crit_point} }}^{{\\rm ext.}}={params[0]:.3f} \\pm {np.sqrt(cov[0,0]):.3f}$ \n $S_0 = {params[1]:.3f} \\pm {np.sqrt(cov[1,1]):.3f}$")

df= pd.DataFrame(df, index=indices)
df_labels= pd.DataFrame(df_labels, index=indices)
df_fitted= pd.DataFrame(df_fitted, index=indices)
df_predicted= pd.DataFrame(df_predicted, index=indices)

#label_pos = {'daac8383bb9bab27dbbd09ebe07bf42c'}
#df = pd.merge(df_params, df, left_index=True, right_index=True)

ed312054ffb1df312e55895d4250c12c
1338435365a19c04a64539d7e75e995e
c1ee62112d7ed57d2e76d6fb2e116255
c0d1499fa767a08d52b42412152bce2e
fb13575a2ea82785862a1dae1795f2c9


In [5]:
df

,entropy,energy,r
ed312054ffb1df312e55895d4250c12c,"[0.6866761672249875, 1.0008305594900366, 1.314...",-151.038511,"[0.14285714285714285, 0.2857142857142857, 0.42..."
1338435365a19c04a64539d7e75e995e,"[0.6875456085521423, 0.8707794972091192, 1.054...",-206.622124,"[0.14285714285714285, 0.2857142857142857, 0.42..."
c1ee62112d7ed57d2e76d6fb2e116255,"[0.6882282252937749, 0.6882483461576544, 0.688...",-32030.183722,"[0.14285714285714285, 0.2857142857142857, 0.42..."
c0d1499fa767a08d52b42412152bce2e,"[0.6882282447967503, 0.6931011773513039, 0.693...",-33.694833,"[0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, ..."
fb13575a2ea82785862a1dae1795f2c9,"[0.6784522915589358, 0.684872716111829, 0.6848...",-33.856866,"[0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, ..."


In [6]:
rows = []

# Loop through each row manually
for idx, row in df.iterrows():
    entropy = row['entropy']
    rs = row['r']
    for i, val in zip(rs, entropy):
        rows.append({'index': idx, 'r': i, 'entropy': val})

# Create the long-format DataFrame
df = pd.DataFrame(rows)
df = df.merge(df_params, left_on='index', right_index=True)


rows = []

# Loop through each row manually
for idx, row in df_fitted.iterrows():
    entropy = row['entropy']
    rs = row['r']
    for i, val in zip(rs, entropy):
        rows.append({'index': idx, 'r': i, 'entropy': val})

df_fitted = pd.DataFrame(rows)
df_fitted = df_fitted.merge(df_params, left_on='index', right_index=True)

rows = []

# Loop through each row manually
for idx, row in df_predicted.iterrows():
    entropy = row['entropy']
    rs = row['r']
    for i, val in zip(rs, entropy):
        rows.append({'index': idx, 'r': i, 'entropy': val})

# Create the long-format DataFrame

df_predicted = pd.DataFrame(rows)
df_predicted = df_predicted.merge(df_params, left_on='index', right_index=True)

df_labels = df_labels.merge(df_params, left_index=True, right_index=True)
df_labels['index'] = df_labels.index

In [7]:
custom_order = jobs
df['index'] = pd.Categorical(df['index'], categories=custom_order, ordered=True)
df.sort_values('index')

,index,r,entropy,sp.simulation_class,sp.sequential.recursive_keys,sp.output_filename,sp.save_psi,sp.model_class,sp.model_params.model_name,sp.model_params.Nf,...,a,g,Nf_num,Nf,Nr_num,$\theta$,\theta,m,diag_method,L
0,ed312054ffb1df312e55895d4250c12c,0.142857,0.686676,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,3,...,0.3,1.0,3,3,4,$\pi$,\pi,"(0.3335,3.0,3.0)",d,32
142,ed312054ffb1df312e55895d4250c12c,20.428571,2.527689,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,3,...,0.3,1.0,3,3,4,$\pi$,\pi,"(0.3335,3.0,3.0)",d,32
143,ed312054ffb1df312e55895d4250c12c,20.571429,2.527689,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,3,...,0.3,1.0,3,3,4,$\pi$,\pi,"(0.3335,3.0,3.0)",d,32
144,ed312054ffb1df312e55895d4250c12c,20.714286,2.527689,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,3,...,0.3,1.0,3,3,4,$\pi$,\pi,"(0.3335,3.0,3.0)",d,32
145,ed312054ffb1df312e55895d4250c12c,20.857143,2.527689,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,3,...,0.3,1.0,3,3,4,$\pi$,\pi,"(0.3335,3.0,3.0)",d,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882,fb13575a2ea82785862a1dae1795f2c9,11.000000,1.810492,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,1,...,0.3,1.0,1,1,4,$\pi$,\pi,(0.3335),d,32
883,fb13575a2ea82785862a1dae1795f2c9,11.200000,1.818377,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,1,...,0.3,1.0,1,1,4,$\pi$,\pi,(0.3335),d,32
884,fb13575a2ea82785862a1dae1795f2c9,11.400000,1.818377,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,1,...,0.3,1.0,1,1,4,$\pi$,\pi,(0.3335),d,32
906,fb13575a2ea82785862a1dae1795f2c9,15.800000,1.831778,GroundStateSearch,"(algorithm_params.trunc_params.chi_max,)",results_simulation.h5,False,MassiveSchwingerModelQubitEncoding,MassiveSchwingerModelVR,1,...,0.3,1.0,1,1,4,$\pi$,\pi,(0.3335),d,32


In [8]:
all_dfs = [df, df_fitted, df_predicted, df_labels]
df_i, df_f, df_p, df_l = [ d for d in all_dfs]

for d in all_dfs:
    # Ensure the column exists before trying to convert it
    if 'index' in d.columns:
        d['index'] = pd.Categorical(d['index'], categories=custom_order, ordered=True)

def col_labeller(labels):
    return f"$\\theta = {"\pi" if np.abs(float(labels) - np.pi)<1e-10 else  "0"}$"
def label_func(key, val):
    if key=='Nf':
        key = "N_f"
    return f"${key}:{val}$"
p = (ggplot(df_i, aes(x='r', y='entropy', color='index' ))
#+ geom_line(size=0.2)
+ geom_point(fill='None', stroke=0.1, size=1, shape='o',show_legend=False)
+ geom_point(data=df_f, stroke=0.1,  shape='.', size=3,show_legend=False)
+ geom_line(data=df_p, color='k', size=0.4)
+facet_grid('. ~ index' )
+ scale_shape_manual(values=['x', '1', '2', '+'])  # Custom shape for each cyl
+ scale_size_manual(values=[2, 2, 2, 0.5])  # Custom shape for each cyl
+ scale_color_manual(values=[ 'purple', 'navy', 'goldenrod', 'teal','slategrey'])
+ geom_text(data=df_l,  mapping=aes(label='label'), y =1,  size=8, color='k')

+ guides(color=guide_legend(override_aes={'size': 1}, nrow=2), size="none") # Adjust marker size for the 'color' legend
+ theme_matplotlib()
+ prl_theme
+ theme(
    figure_size=(7, 1.8),  # Width x height in inches
    strip_background=element_blank(), # Removes the boxes
    strip_text=element_blank()        # Removes the text labels
    )
+labs(#title='Groundstate Energy $E_0$ vs $m_1=m_2$',
         y='$S_\mathrm{A}$',
         x='$r$',
         color='',
         shape=''
         )
)
ggsave(p, "../../../plots/"+datetime.today().strftime('%Y-%m-%d') + "_Plot_Ising_missmatch.pdf")
p

<>:10: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:35: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:10: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:35: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
/tmp/ipykernel_3647722/2581420109.py:10: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
/tmp/ipykernel_3647722/2581420109.py:35: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
/local/d